# 11b · Reprodução de Lira et al.: por que o código público não reproduz as tabelas

Na calibração (notebook 11), as três receitas fiéis ao código público (`B`, `B_smoteenn` e `A`) produziram, em **todas** as 14 unidades rodadas, redes que predizem uma classe só, inclusive nas ResNets pré-treinadas, que no artigo chegam a 0,82–0,87 de acurácia em entrada/saída.

Hipótese: o skorch **não embaralha** os lotes por padrão, e o CSV do dataset vem **ordenado por classe** (42 encostado, 1.609 distância, 232 curta, 668 saídas). Cada lote de treino tem, então, uma classe só.

Este notebook testa a hipótese com três variantes que **não estão no código público**, sempre na partição com vazamento:

| Variante | O que muda |
|---|---|
| `B_embaralhada` | receita B com `iterator_train__shuffle=True` (lotes sorteados a cada época) |
| `B_permutada` | receita B com as linhas de treino permutadas **uma vez** (como se o CSV não estivesse ordenado por classe) |
| `A_embaralhada` | receita A (SMOTEENN, pesos balanceados, cosseno, 1000 épocas) com lotes embaralhados |

Nada aqui decide a receita: o resultado é mostrado ao lado das linhas publicadas e gravado em `resultados/diagnostico_lira.json`. Precisão (AMP) e épocas da receita B (300) são as que a calibração já tinha fixado.

**Como rodar:** **Executar tudo**; se cair, executar tudo de novo. GPU com **RAM alta**.

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
RAIZ_TRABALHO = "/content"
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_MODULO_ESPERADO = "59f5fafb471aa153a4df440b1857eb3d365e39bde8835f178406047291adfce0"   # reproducao_lira.py
SHA_TABELAS_ESPERADO = "74cb1aedf91057c2cbf92896d02759a6d992bf7962e2ee18eb07f2b16d616d7c"  # dados/lira2024_tabelas_completas.csv
ABA = "A"
PRECISAO, EPOCAS_B = "amp", 300          # fixados nas etapas 2 e 3 do notebook 11
# (receita, modelos): as mais baratas primeiro
PLANO = [
    ("B_embaralhada", ["Resnet18", "Alexnet", "Resnet152"]),
    ("B_permutada", ["Resnet18", "Alexnet", "Resnet152"]),
    ("A_embaralhada", ["Resnet18", "Alexnet"]),
]

In [ ]:
# Versões do environment.yml do leguwoi onde o Colab permite; o que não instalar fica na versão do
# Colab e isso é registrado em cada resultado.json (campo "ambiente"). Rode ANTES de importar.
import subprocess, sys
def pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
pip("scikit-multilearn==0.2.0")
FIXOS = {
    "albumentations": ("1.4.0", "import numpy as np, albumentations as A; from albumentations.pytorch import ToTensorV2; "
        "A.Compose([A.RGBShift(p=1), A.UnsharpMask(p=1), A.SafeRotate(limit=30, p=1), A.GaussNoise(p=1), A.ISONoise(p=1), "
        "A.Sharpen(p=1), A.CLAHE(p=1), A.ElasticTransform(p=1), A.Resize(224, 224), A.Normalize(), ToTensorV2()])"
        "(image=np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8))"),
    "imbalanced-learn": ("0.12.0", "import numpy as np; from imblearn.combine import SMOTEENN; "
        "SMOTEENN(random_state=64).fit_resample(np.random.rand(80, 6), np.array([0] * 60 + [1] * 20))"),
    "skorch": ("0.15.0", "import numpy as np, torch, skorch; from skorch import NeuralNetClassifier; "
        "NeuralNetClassifier(torch.nn.Linear, module__in_features=4, module__out_features=2, max_epochs=1, verbose=0, "
        "criterion=torch.nn.CrossEntropyLoss, train_split=skorch.dataset.ValidSplit(cv=5, stratified=True))"
        ".fit(np.random.rand(40, 4).astype('float32'), np.array([0, 1] * 20))"),
}
for pacote, (versao, teste) in FIXOS.items():
    ok = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pacote}=={versao}"]).returncode == 0
    ok = ok and subprocess.run([sys.executable, "-c", teste], capture_output=True).returncode == 0
    if not ok:
        print(f"⚠️ {pacote}=={versao} não funciona neste Colab; instalando a versão atual (fica registrado).")
        pip("-U", pacote)
        r = subprocess.run([sys.executable, "-c", teste], capture_output=True, text=True)
        assert r.returncode == 0, f"{pacote} não funciona nem na versão atual:\n{r.stderr[-800:]}"
    else:
        print(f"{pacote}=={versao} OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, "O protocolo_comum.py do Drive não é o publicado."
ARQ_MODULO = f"{DIR_DRIVE}/reproducao_lira.py"
assert pc.sha256_arquivo(ARQ_MODULO) == SHA_MODULO_ESPERADO, (
    "O reproducao_lira.py do Drive não é o do repositório (SHA diferente). Envie de novo o arquivo certo.")
import reproducao_lira as rl
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar tipo → GPU (qualquer uma serve aqui)."
pc.definir_id_sessao(ABA)
print("GPU:", torch.cuda.get_device_name(0), "| módulo", rl.VERSAO, SHA_MODULO_ESPERADO[:16], "… | dono das travas:", pc.ID_SESSAO)

# Clona dataset e leguwoi nos commits fixados, importa o código do leguwoi e confere as sementes dele
ctx = rl.preparar(RAIZ_TRABALHO, caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}")
print("Sementes no leguwoi:", *ctx.meta["sementes_no_leguwoi"], sep="\n  ")
print("V_0.1 × master:", json.dumps(ctx.meta["diferencas_v01"], indent=1))
erros = rl.testar_metricas()
assert not erros, erros
print("Partição com vazamento = a do 04b:", rl.conferir_com_holdout_do_drive(ctx, f"{DIR_DRIVE}/dados/holdout_lira2024.csv"))
display(pd.DataFrame([rl.resumo_particao(ctx, t, p) for t in rl.TAREFAS for p in ("vazado_p42", "agrupado_d0")]))
print("Ambiente:", rl.ambiente())

R = f"{DIR_DRIVE}/resultados/reproducao_lira_calibracao"
C = f"{DIR_DRIVE}/checkpoints/reproducao_lira_calibracao"
ARQ_TAB = f"{DIR_DRIVE}/dados/lira2024_tabelas_completas.csv"
assert pc.sha256_arquivo(ARQ_TAB) == SHA_TABELAS_ESPERADO
PUB = pd.read_csv(ARQ_TAB).set_index(["tarefa", "modelo_leguwoi"])

def quadro():
    d = rl.ler_resultados(R)
    d = d[(d["particao"] == "vazado_p42") & d["receita"].isin(["A", "B", "B_smoteenn", *rl.DIAGNOSTICOS])]
    d = d[["tarefa", "modelo", "receita", "acc", "f1_M", "auc_M", "epocas", "treino_min"]].copy()
    for c in ("acc", "f1_M", "auc_M"):
        d[f"pub_{c}"] = [PUB.loc[(t, m), c] for t, m in zip(d["tarefa"], d["modelo"])]
    d["uma_classe_so"] = (d["auc_M"] - 0.5).abs() < 0.005
    return d.sort_values(["tarefa", "modelo", "receita"]).reset_index(drop=True)

## Rodada

In [ ]:
for receita, modelos in PLANO:
    kw = {"epocas": EPOCAS_B} if not receita.startswith("A") else {}
    rl.executar_fila(ctx, rl.montar_fila(receita, modelos=modelos, particoes=("vazado_p42",)), R, C, precisao=PRECISAO, **kw)
    display(quadro().round(3))

## Resultado

In [ ]:
q = quadro()
display(q.round(3))
resumo = (q.assign(erro=lambda d: ((d["acc"] - d["pub_acc"]).abs() + (d["f1_M"] - d["pub_f1_M"]).abs()
                                   + (d["auc_M"] - d["pub_auc_M"]).abs()) / 3)
            .groupby("receita").agg(unidades=("acc", "size"), uma_classe_so=("uma_classe_so", "sum"),
                                    erro_abs_medio=("erro", "mean")).sort_values("erro_abs_medio"))
display(resumo.round(3))
pc.gravar_json_atomico({"gerado_em": pc.agora_iso(), "modulo_sha256": SHA_MODULO_ESPERADO, "precisao": PRECISAO,
                        "epocas_B": EPOCAS_B, "quadro": q.to_dict("records"),
                        "resumo": resumo.reset_index().to_dict("records"), "ambiente": rl.ambiente()},
                       Path(DIR_DRIVE) / "resultados" / "diagnostico_lira.json")
print("Gravado: resultados/diagnostico_lira.json")

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*).